# HunyuanOCR — bản Colab A100 (vLLM, tốc độ tối đa)

OCR keyframe AIC bằng **`tencent/HunyuanOCR`** trên **1× A100 40GB** + **vLLM batched**.
Nhanh hơn ~một bậc so với 2×T4. Resume bằng cách đọc các file `*.gpu*.jsonl` đã process (bạn upload lên `/content`).

### Trước khi chạy
1. **Runtime → Change runtime type → A100 GPU.**
2. Upload dữ liệu lên Colab:
   - Ảnh: `/content/keyframes/Keyframes_K01/...` (như trong file browser của bạn).
   - File đã process (để skip): các `K08.gpu0.jsonl`, `L21.gpu1.jsonl`, ... đặt ở `/content` (hoặc 1 thư mục, khai trong `RESUME_DIRS`).
3. (Khuyến nghị) Mount Google Drive để **lưu output** khỏi mất khi Colab ngắt — xem cell Mount Drive.

### Thứ tự: CONFIG → Install → (Mount Drive) → Helpers/Discovery → Load model → Run → Post-process

In [ ]:
# ===================== CONFIG (Colab A100) — chỉ sửa ở đây =====================
CATEGORIES = "ALL"              # "ALL" hoặc list ["K01","K02","L21"]; tiền tố "L26" khớp L26_a..e

# ---- đường dẫn trên Colab ----
INPUT_ROOT   = "/content"                 # nơi DÒ "Keyframes_*" (vd /content/keyframes/Keyframes_K01)
RESUME_DIRS  = ["/content"]               # nơi chứa *.gpu0/.gpu1.jsonl đã process (upload lên)
OUT_DIR      = "/content/ocr_out"         # output mới (NÊN trỏ vào Drive, xem cell Mount Drive)

# ---- A100: đẩy tối đa tốc độ ----
ENGINE        = "vllm"          # A100: vLLM nhanh nhất (đã xác nhận chạy). Lỗi CUDA -> đổi "transformers"
PROMPT_MODE   = "spotting"      # "spotting" (chữ + toạ độ) | "plain" (chỉ chữ)
DTYPE         = "bfloat16"      # A100 có bf16 tensor-core (đúng precision gốc model)
BATCH_SIZE    = 8              # vision-heavy: batch lớn dễ làm vLLM OOM-crash. Ổn rồi mới tăng 16/32
GPU_MEM_UTIL  = 0.85           # chừa chỗ cho activation thị giác (cao quá -> engine OOM-crash)
MAX_MODEL_LEN = 8192
ENFORCE_EAGER = True           # ổn định hơn (False nhanh hơn chút nhưng dễ crash với multimodal)
IMG_MAX_SIDE  = 1280           # GIỮ NGUYÊN (chất lượng). Hạ = nhanh hơn nhưng mất chữ nhỏ
MAX_NEW_TOKENS = 2048          # trần; model gặp EOS tự dừng sớm

RETRY_ERRORS = False
IMG_EXTS     = (".jpg", ".jpeg", ".png", ".webp")
MODEL_ID     = "tencent/HunyuanOCR"

PROMPTS = {
    "spotting": "检测并识别图片中的文字，将文本坐标格式化输出。",
    "plain":    "识别图片中的所有文字，按从上到下、从左到右的阅读顺序输出纯文本。",
}
# ==============================================================================
print("CONFIG OK | A100 vLLM | batch =", BATCH_SIZE, "| dtype =", DTYPE)

In [ ]:
# ===================== Install (Colab) =====================
import sys, subprocess, importlib.util
def _pip(*a):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a], check=True)

_pip("tqdm")
if ENGINE == "vllm":
    if importlib.util.find_spec("vllm") is None:
        print("Cài vLLM (gỡ torch mặc định của Colab để vLLM tự kéo torch+CUDA khớp)...")
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y",
                        "vllm", "torch", "torchvision", "torchaudio"])
        _pip("vllm")
        print("✅ Cài vLLM xong.")
        print("⚠️ BÂY GIỜ: Runtime → Restart runtime, rồi chạy lại TỪ CONFIG (cell này tự bỏ qua lần 2).")
    else:
        print("✅ vLLM đã sẵn sàng — bỏ qua cài đặt.")
else:
    _pip("git+https://github.com/huggingface/transformers@82a06db03535c49aa987719ed0746a76093b1ec4")
    _pip("accelerate", "pillow")
    print("✅ Cài transformers (HunyuanOCR) xong. ENGINE = transformers.")


## Tải keyframe từ Hugging Face → `/content/keyframes`

Chạy 1 lần mỗi session (Colab xoá `/content` khi ngắt). Bỏ qua nếu `/content/keyframes` đã có sẵn.

In [ ]:
# ===================== Tải keyframe từ Hugging Face -> /content/keyframes =====================
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub", "hf_xet"], check=True)
import os
from getpass import getpass
from pathlib import Path
from huggingface_hub import HfApi, hf_hub_download, list_repo_files

REPO_ID   = "Baonenha1/AIC_Keyframe_K"
REPO_TYPE = "dataset"
DOWNLOAD_DIR = Path("/content/hf_keyframe_tars")
EXTRACT_DIR  = Path("/content/keyframes")
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"

hf_token = getpass("HF token (READ): ").strip()
assert hf_token.startswith("hf_"), "Token sai format (phải bắt đầu bằng hf_)"
print("Logged in as:", HfApi().whoami(token=hf_token)["name"])

DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
tars = sorted(f for f in list_repo_files(REPO_ID, repo_type=REPO_TYPE, token=hf_token)
              if f.startswith("keyframes/") and f.endswith(".tar"))
print("Tìm thấy", len(tars), "tar shard")
for rf in tars:
    print("↓", rf)
    tp = Path(hf_hub_download(REPO_ID, repo_type=REPO_TYPE, filename=rf,
                              local_dir=str(DOWNLOAD_DIR), token=hf_token))
    subprocess.run(["tar", "-xf", str(tp), "-C", str(EXTRACT_DIR)], check=True)
    tp.unlink()   # xoá tar sau khi giải nén cho nhẹ đĩa
print("✅ Done ->", EXTRACT_DIR)

## (Tuỳ chọn nhưng nên làm) Mount Google Drive để giữ output

Colab xoá `/content` khi ngắt. Mount Drive rồi trỏ `OUT_DIR` vào đó để **không mất tiến độ** và **resume xuyên session**.

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")
# OUT_DIR = "/content/drive/MyDrive/aic_ocr_out"   # output lưu vào Drive
# RESUME_DIRS = RESUME_DIRS + [OUT_DIR]            # lần sau tự resume từ Drive
# import os; os.makedirs(OUT_DIR, exist_ok=True)
# print("OUT_DIR =", OUT_DIR)

In [ ]:
# ===================== Helpers + Discovery =====================
import os, json, time, glob, unicodedata, re as _re
from pathlib import Path

def normalize_nfc(s):
    return unicodedata.normalize("NFC", s) if s else s

def fold_diacritics(s):
    if not s:
        return s
    d = unicodedata.normalize("NFD", s)
    d = "".join(c for c in d if unicodedata.category(c) != "Mn")
    d = d.replace("đ", "d").replace("Đ", "D")
    return unicodedata.normalize("NFC", d).lower()

_SPOT_PAT = _re.compile(r"(.*?)\(\s*(\d+)\s*,\s*(\d+)\s*\)\s*,\s*\(\s*(\d+)\s*,\s*(\d+)\s*\)", _re.S)
def parse_spotting(raw):
    items, last = [], 0
    for m in _SPOT_PAT.finditer(raw or ""):
        txt = m.group(1).strip()
        if txt:
            items.append({"text": txt, "box": [int(m.group(2)), int(m.group(3)),
                                                int(m.group(4)), int(m.group(5))]})
        last = m.end()
    tail = (raw or "")[last:].strip()
    if tail:
        items.append({"text": tail, "box": None})
    plain = " ".join(it["text"] for it in items) if items else (raw or "").strip()
    return items, plain

def find_keyframe_dirs(root, prefix="Keyframes_", max_depth=6):
    found = {}
    def walk(p, depth):
        if depth > max_depth:
            return
        try:
            entries = list(os.scandir(p))
        except (PermissionError, FileNotFoundError):
            return
        for e in entries:
            if e.is_dir(follow_symlinks=False):
                if e.name.startswith(prefix):
                    found.setdefault(e.name[len(prefix):], e.path)
                else:
                    walk(e.path, depth + 1)
    walk(root, 0)
    return dict(sorted(found.items()))

def list_images(cat_root, exts):
    imgs = []
    for dp, _, fs in os.walk(cat_root):
        for f in fs:
            if f.lower().endswith(exts):
                imgs.append(os.path.join(dp, f))
    imgs.sort()
    return imgs

def make_id(label, img_path):
    p = Path(img_path)
    return f"{label}/{p.parent.name}/{p.stem}"

def resolve_categories(req, available):
    if req == "ALL" or req == ["ALL"]:
        return list(available)
    req = req if isinstance(req, list) else [req]
    sel = []
    for r in req:
        if r in available:
            sel.append(r)
        else:
            pref = [l for l in available if l.startswith(r)]
            sel += pref if pref else (print("⚠️ không thấy loại:", r) or [])
    seen, out = set(), []
    for s in sel:
        if s not in seen:
            seen.add(s); out.append(s)
    return out

AVAILABLE = find_keyframe_dirs(INPUT_ROOT)
print(f"Tìm thấy {len(AVAILABLE)} loại keyframe trong {INPUT_ROOT}:")
for label, path in AVAILABLE.items():
    print(f"  {label:10s} -> {path}")
print("\nSẽ chạy:", resolve_categories(CATEGORIES, AVAILABLE))

In [ ]:
# ===================== Load model + ocr_batch() (engine-aware) =====================
import torch
from PIL import Image
PROMPT = PROMPTS[PROMPT_MODE]
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU?!")

def load_image(path, max_side):
    im = Image.open(path).convert("RGB")
    if max_side:
        w, h = im.size
        m = max(w, h)
        if m > max_side:
            s = max_side / m
            im = im.resize((max(1, int(w * s)), max(1, int(h * s))))
    return im

if ENGINE == "vllm":
    from vllm import LLM, SamplingParams
    from transformers import AutoProcessor
    llm = LLM(model=MODEL_ID, trust_remote_code=True, dtype=DTYPE,
              gpu_memory_utilization=GPU_MEM_UTIL, max_model_len=MAX_MODEL_LEN,
              limit_mm_per_prompt={"image": 1}, enforce_eager=ENFORCE_EAGER)
    processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
    _sp = SamplingParams(temperature=0.0, max_tokens=MAX_NEW_TOKENS)
    def _build(img, prompt_text):
        messages = [{"role": "system", "content": ""},
                    {"role": "user", "content": [{"type": "image", "image": ""},
                                                  {"type": "text", "text": prompt_text}]}]
        p = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        return {"prompt": p, "multi_modal_data": {"image": [img]}}
    def ocr_batch(images, prompt=PROMPT):
        outs = llm.generate([_build(im, prompt) for im in images], _sp, use_tqdm=False)
        return [o.outputs[0].text.strip() for o in outs]

else:  # transformers — chắc chạy trên Colab; A100 vẫn nhanh hơn T4 nhiều (bf16 tensor-core + sdpa)
    from transformers import AutoProcessor, HunYuanVLForConditionalGeneration
    processor = AutoProcessor.from_pretrained(MODEL_ID, use_fast=False, trust_remote_code=True)
    try:
        model = HunYuanVLForConditionalGeneration.from_pretrained(
            MODEL_ID, attn_implementation="sdpa", dtype=torch.bfloat16,
            device_map={"": 0}, trust_remote_code=True)
    except Exception as _e:
        print("sdpa fail -> eager:", _e)
        model = HunYuanVLForConditionalGeneration.from_pretrained(
            MODEL_ID, attn_implementation="eager", dtype=torch.bfloat16,
            device_map={"": 0}, trust_remote_code=True)
    model.eval()
    @torch.no_grad()
    def _ocr_one(img, prompt_text):
        messages = [{"role": "user", "content": [{"type": "image", "image": img},
                                                 {"type": "text", "text": prompt_text}]}]
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=text, images=img, return_tensors="pt").to(model.device)
        gen = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
        n = inputs["input_ids"].shape[1]
        return processor.decode(gen[0][n:], skip_special_tokens=True).strip()
    def ocr_batch(images, prompt=PROMPT):
        return [_ocr_one(im, prompt) for im in images]   # 1 ảnh/lần

# sanity test 1 ảnh
_probe = next((list_images(p, IMG_EXTS)[0] for p in AVAILABLE.values() if list_images(p, IMG_EXTS)), None)
if _probe:
    print("Test:", _probe)
    print("→", ocr_batch([load_image(_probe, IMG_MAX_SIDE)])[0][:400])
else:
    print("⚠️ Chưa thấy ảnh — kiểm tra INPUT_ROOT / cấu trúc thư mục.")


In [ ]:
# ===================== Run (batched + resume từ shard đã upload) =====================
from tqdm.auto import tqdm
os.makedirs(OUT_DIR, exist_ok=True)
_ERR = "__OCR_ERROR__:"

def make_record(label, idv, ipath, raw, err=None):
    if err is not None:
        return {"id": idv, "category": label, "video": idv.split("/")[1], "image_path": ipath,
                "status": "error", "error": str(err)[:500], "raw": "", "boxes": [],
                "text_nfc": "", "text_fold": "", "ts": time.time()}
    raw = raw.strip()
    items, plain = parse_spotting(raw) if PROMPT_MODE == "spotting" else ([], raw)
    return {"id": idv, "category": label, "video": idv.split("/")[1], "image_path": ipath,
            "status": "ok", "error": "", "raw": raw, "boxes": items,
            "text_nfc": normalize_nfc(plain), "text_fold": fold_diacritics(plain), "ts": time.time()}

def _resume_paths(label):
    # CHỈ glob nông (top-level + 1 cấp) — KHÔNG recursive, tránh quét cả cây keyframes (treo 20+ phút!)
    paths = set()
    for d in [OUT_DIR] + list(RESUME_DIRS):
        paths |= set(glob.glob(os.path.join(d, f"{label}*.jsonl")))      # /content/K01*.jsonl
        paths |= set(glob.glob(os.path.join(d, "*", f"{label}*.jsonl")))  # /content/<sub>/K01*.jsonl (1 cấp)
    return sorted(paths)

def load_done(label):
    done = set()
    for jp in _resume_paths(label):
        try:
            with open(jp, encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        r = json.loads(line)
                    except json.JSONDecodeError:
                        continue
                    if RETRY_ERRORS and r.get("status") == "error":
                        continue
                    done.add(r["id"])
        except FileNotFoundError:
            pass
    return done

def process_category(label, root):
    imgs = list_images(root, IMG_EXTS)
    done = load_done(label)
    todo = [p for p in imgs if make_id(label, p) not in done]
    print(f"[{label}] tổng {len(imgs)} | đã xong {len(done)} | còn {len(todo)}")
    if not todo:
        return
    out_path = os.path.join(OUT_DIR, f"{label}.jsonl")
    with open(out_path, "a", encoding="utf-8") as fout:
        pbar = tqdm(range(0, len(todo), BATCH_SIZE), desc=label)
        for i in pbar:
            chunk = todo[i:i + BATCH_SIZE]
            pil, metas = [], []
            for p in chunk:
                idv = make_id(label, p)
                try:
                    pil.append(load_image(p, IMG_MAX_SIDE)); metas.append((idv, p))
                except Exception as e:
                    fout.write(json.dumps(make_record(label, idv, p, "", err=f"load:{e}"), ensure_ascii=False) + "\n")
            if not pil:
                continue
            try:
                raws = ocr_batch(pil)
            except Exception as e:
                print(f"\n⚠️ batch lỗi (sẽ retry ở lần chạy sau): {e}")
                continue   # KHÔNG ghi -> các ảnh này vẫn 'chưa xong' -> tự chạy lại
            for (idv, p), raw in zip(metas, raws):
                fout.write(json.dumps(make_record(label, idv, p, raw), ensure_ascii=False) + "\n")
            fout.flush()
            pbar.set_postfix(done=i + len(chunk))
    print(f"[{label}] ✅ {out_path}")

selected = resolve_categories(CATEGORIES, AVAILABLE)
print("Chạy:", selected, "\n")
t0 = time.time()
for label in selected:
    process_category(label, AVAILABLE[label])
print(f"\n=== HOÀN TẤT trong {(time.time()-t0)/60:.1f} phút ===")

In [ ]:
# ===================== Xem thử + thống kê =====================
import glob, json
files = sorted(glob.glob(os.path.join(OUT_DIR, "*.jsonl")))
total = ok = err = 0
for fp in files:
    with open(fp, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                total += 1
                r = json.loads(line)
                ok += r.get("status") == "ok"; err += r.get("status") == "error"
print(f"{len(files)} file | {total} records | ok={ok} | error={err}")
for fp in files:
    print("  ", fp)

selected = resolve_categories(CATEGORIES, AVAILABLE)
if selected:
    fp = os.path.join(OUT_DIR, f"{selected[0]}.jsonl")
    if os.path.exists(fp):
        rows = [json.loads(l) for l in open(fp, encoding="utf-8") if l.strip()]
        for r in [x for x in rows if x.get("status") == "ok"][:3]:
            print("\n" + "=" * 60)
            print("ID :", r["id"], "| n_boxes:", len(r.get("boxes", [])))
            print(r["text_nfc"][:600])

In [ ]:
# ===================== (POST-PROCESS) Dọn nhiễu logo/giờ/watermark -> text_clean =====================
import os, re, glob, json, unicodedata
CLEAN_PATH = os.path.join(os.path.dirname(OUT_DIR) or ".", "ocr_clean.jsonl")

def _nfc(s):  return unicodedata.normalize("NFC", s) if s else s
def _fold(s):
    if not s: return s
    d = unicodedata.normalize("NFD", s)
    d = "".join(c for c in d if unicodedata.category(c) != "Mn")
    return unicodedata.normalize("NFC", d.replace("đ", "d").replace("Đ", "D")).lower()

_TIME  = re.compile(r"^\d{1,2}:\d{2}(:\d{2})?$")
_LOGOW = {"htv", "htv7", "hd", "giay", "subscribe", "subscribed"}
def _is_noise(text):
    s = (text or "").strip()
    if not s or len(s) == 1 or _TIME.match(s):
        return True
    w = _fold(s).split()
    return bool(w) and all(x in _LOGOW or len(x) == 1 or _TIME.match(x) for x in w)
def clean_text(boxes):
    return " ".join(b.get("text", "").strip() for b in (boxes or []) if not _is_noise(b.get("text", "")))

# gom từ CẢ ocr_out (mới) LẪN shard upload ở RESUME_DIRS — nông, KHÔNG recursive
_src = set(glob.glob(os.path.join(OUT_DIR, "*.jsonl")))
for _d in globals().get("RESUME_DIRS", []):
    _src |= set(glob.glob(os.path.join(_d, "*.jsonl")))
    _src |= set(glob.glob(os.path.join(_d, "*", "*.jsonl")))
records = {}
for fp in sorted(_src):
    for line in open(fp, encoding="utf-8"):
        line = line.strip()
        if not line: continue
        try: r = json.loads(line)
        except json.JSONDecodeError: continue
        rid = r.get("id")
        if rid and (rid not in records or r.get("ts", 0) >= records[rid].get("ts", 0)):
            records[rid] = r

n = cb = ca = 0
with open(CLEAN_PATH, "w", encoding="utf-8") as fout:
    for r in records.values():
        if r.get("status") != "ok": continue
        n += 1
        cl = clean_text(r.get("boxes"))
        r["text_clean"] = _nfc(cl); r["text_clean_fold"] = _fold(cl)
        cb += len(r.get("text_nfc", "")); ca += len(cl)
        fout.write(json.dumps(r, ensure_ascii=False) + "\n")
print(f"{len(records)} record (ok={n}) -> {CLEAN_PATH} | giảm {100*(1-ca/max(1,cb)):.1f}% nhiễu")

## Ghi chú

- **Tốc độ:** vLLM batched trên A100 nhanh hơn 2×T4 (transformers, 1 ảnh/lần) ~một bậc độ lớn. OOM thì hạ `BATCH_SIZE` (64→32→16). Khởi tạo lỗi/treo thì đặt `ENFORCE_EAGER = True`.
- **Chất lượng KHÔNG đổi** so với T4: cùng model, `bfloat16`, `IMG_MAX_SIDE=1280`, greedy. Đừng hạ ảnh / token nếu muốn giữ độ chính xác.
- **Resume:** đọc mọi `*.jsonl` (kể cả shard `.gpu0/.gpu1` bạn upload) trong `OUT_DIR` + `RESUME_DIRS`, skip theo field `id`. Output mới ghi `OUT_DIR/<loại>.jsonl`.
- **Giữ tiến độ:** mount Drive + trỏ `OUT_DIR` vào Drive (cell Mount Drive), thêm `OUT_DIR` vào `RESUME_DIRS` cho lần sau.
- **Nếu `AutoProcessor` lỗi** (thiếu processor tiếng Việt): chạy `!pip install -q "git+https://github.com/huggingface/transformers@82a06db03535c49aa987719ed0746a76093b1ec4"` rồi Restart + chạy lại.
- **vLLM message format chưa chạy thật trên Hunyuan** — chạy cell Load model (test 1 ảnh) trước; nếu lỗi `_build`, gửi log để chỉnh.


### Muốn dùng vLLM (nhanh hơn) dù gặp `libcudart.so.13`?
vLLM build cho CUDA khác CUDA của Colab. Thử reinstall sạch để vLLM tự kéo torch + CUDA libs khớp, rồi **Restart runtime**:
```python
!pip uninstall -y vllm torch torchvision torchaudio
!pip install -q vllm
# Runtime → Restart runtime → chạy lại từ CONFIG với ENGINE="vllm"
```
Nếu vẫn lỗi (driver A100 của Colab không đủ mới cho CUDA mà vLLM cần) → quay lại `ENGINE="transformers"` (vẫn nhanh trên A100).
